In [36]:
import transformers
import datasets
import torch
import modelscope
print(transformers.__version__)
print(datasets.__version__)
print(torch.__version__)
print(modelscope.__version__)
import numpy as np
import torchaudio
from pathlib import Path
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
from pathlib import Path
import librosa
import numpy as np
import torchaudio
import random
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import pandas as pd
import contextlib
import os
from collections import Counter
import torch
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks
# from modelscope.models import Model
# from funasr import AutoModel


4.23.1
3.6.0
2.2.1+cpu
1.29.2


In [13]:
import os
import pandas as pd

Crema = r"C:\Users\havei\Desktop\crema\AudioWAV" #需動
print(Crema)

crema_directory_list = os.listdir(Crema)
print("找到的檔案數：", len(crema_directory_list))

file_emotion = []
file_path = []

# 3. 處理每個檔案
for file in crema_directory_list:

    file_path.append(os.path.join(Crema, file))
    part = file.split('_')

    if part[2] == 'SAD':
        file_emotion.append('sad')
    elif part[2] == 'ANG':
        file_emotion.append('angry')
    elif part[2] == 'DIS':
        file_emotion.append('disgust')
    elif part[2] == 'FEA':
        file_emotion.append('fear')
    elif part[2] == 'HAP':
        file_emotion.append('happy')
    elif part[2] == 'NEU':
        file_emotion.append('neutral')
    else:
        file_emotion.append('Unknown')

# 4. 建 DataFrame
emotion_df = pd.DataFrame(file_emotion, columns=['Emotions'])
path_df = pd.DataFrame(file_path, columns=['Path'])
Crema_df = pd.concat([emotion_df, path_df], axis=1)

print(Crema_df.head(10))

C:\Users\havei\Desktop\crema\AudioWAV
找到的檔案數： 7442
  Emotions                                               Path
0    angry  C:\Users\havei\Desktop\crema\AudioWAV\1001_DFA...
1  disgust  C:\Users\havei\Desktop\crema\AudioWAV\1001_DFA...
2     fear  C:\Users\havei\Desktop\crema\AudioWAV\1001_DFA...
3    happy  C:\Users\havei\Desktop\crema\AudioWAV\1001_DFA...
4  neutral  C:\Users\havei\Desktop\crema\AudioWAV\1001_DFA...
5      sad  C:\Users\havei\Desktop\crema\AudioWAV\1001_DFA...
6    angry  C:\Users\havei\Desktop\crema\AudioWAV\1001_IEO...
7    angry  C:\Users\havei\Desktop\crema\AudioWAV\1001_IEO...
8    angry  C:\Users\havei\Desktop\crema\AudioWAV\1001_IEO...
9  disgust  C:\Users\havei\Desktop\crema\AudioWAV\1001_IEO...


In [37]:
print(sorted(Crema_df['Emotions'].unique()))

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']


In [40]:
# ------------------------ 基本設定 ------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model_path = r"C:\Users\havei\Desktop\voice\Model_5s2(89%).pth"  # <- 改這個
Crema = r"C:\Users\havei\Desktop\crema\AudioWAV"  # <- 改這個
sr = 16000
target_len = sr * 5
batch_size = 16

# ------------------------ Emotion2Vec 初始化 ------------------------
feature_model = pipeline(
    task=Tasks.emotion_recognition,
    model="iic/emotion2vec_base")

# ------------------------ 建立 DataFrame ------------------------
crema_directory_list = os.listdir(Crema)
file_emotion = []
file_path = []

for file in crema_directory_list:
    file_path.append(os.path.join(Crema, file))
    part = file.split('_')
    emo = part[2]
    if emo == 'SAD':
        file_emotion.append('sad')
    elif emo == 'ANG':
        file_emotion.append('angry')
    elif emo == 'DIS':
        file_emotion.append('disgust')
    elif emo == 'FEA':
        file_emotion.append('fear')
    elif emo == 'HAP':
        file_emotion.append('happy')
    elif emo == 'NEU':
        file_emotion.append('neutral')
    else:
        file_emotion.append('Unknown')

emotion_df = pd.DataFrame(file_emotion, columns=['Emotions'])
path_df = pd.DataFrame(file_path, columns=['Path'])
Crema_df = pd.concat([emotion_df, path_df], axis=1)

# ------------------------ Dataset ------------------------
class VoiceDataset(Dataset):
    def __init__(self, df, sr=16000, target_len=80000):
        self.df = df
        self.sr = sr
        self.target_len = target_len

    def __len__(self):
        return len(self.df)

    def pad_or_trim(self, wav):
        L = wav.shape[-1]
        if L >= self.target_len:
            start = np.random.randint(0, L - self.target_len + 1)
            return wav[..., start:start + self.target_len]
        else:
            pad = self.target_len - L
            return F.pad(wav, (0, pad))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav_path = row['Path']
        label = row['Emotions']

        wav, sr0 = torchaudio.load(wav_path)
        wav = wav.squeeze(0)

        if sr0 != self.sr:
            res = torchaudio.transforms.Resample(sr0, self.sr)
            wav = res(wav)

        wav = wav.float()
        wav = wav / (wav.abs().max() + 1e-8)
        wav = self.pad_or_trim(wav)

        return wav, label

test_dataset = VoiceDataset(Crema_df, sr=sr, target_len=target_len)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ------------------------ Emotion2Vec embedding ------------------------
def extract_features_batch(waveforms):
    feats_list = []
    for wav in waveforms:
        wav = wav.unsqueeze(0)
        with torch.no_grad():
            with open(os.devnull, 'w') as fnull:
                with contextlib.redirect_stderr(fnull):
                    out = feature_model(wav)
        feats_np = out[0]['feats']
        feats = torch.from_numpy(feats_np).float()
        feats_list.append(feats)
    return torch.stack(feats_list).to(device)

# ------------------------ Classifier ------------------------
def build_classifier(num_classes=6):
    return nn.Sequential(
        nn.BatchNorm1d(768),
        nn.Dropout(0.1),
        nn.Linear(768, 512),
        nn.GELU(),
        nn.BatchNorm1d(512),
        nn.Dropout(0.1),
        nn.Linear(512, num_classes)
    ).to(device)

# 如果你的 model.pth 是整個模型檔
classifier = torch.load(model_path, map_location=device)
classifier.to(device)
classifier.eval()
print("Classifier loaded.")

# ------------------------ Label mapping ------------------------
label_order = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']
label2id = {l: i for i, l in enumerate(label_order)}
id2label = {i: l for i, l in enumerate(label_order)}

# ------------------------ 推論 ------------------------
pred_all = []
true_all = []

with torch.no_grad():
    for waveforms, labels in tqdm(test_loader):
        waveforms = waveforms.to(device)
        feats = extract_features_batch(waveforms)
        outputs = classifier(feats)

        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        trues = []
        for l in labels:
            trues.append(label2id[l])

        pred_all.extend(preds[:len(trues)])
        true_all.extend(trues)

# ------------------------ Accuracy ------------------------
pred_all = np.array(pred_all)
true_all = np.array(true_all)
acc = (pred_all == true_all).mean() * 100
print(f"\n⭐ Test Accuracy = {acc:.2f}%")

test_acc_list = [] 
test_acc_list.append(acc) 

Using device: cpu


2025-11-20 12:37:26,636 - modelscope - WARNING - Model revision not specified, use revision: v2.0.4


2025-11-20 12:37:31,290 - modelscope - WARNING - Model revision not specified, use revision: v2.0.4
2025-11-20 12:37:31,672 - modelscope - INFO - initiate model from C:\Users\havei\.cache\modelscope\hub\models\iic\emotion2vec_base
2025-11-20 12:37:31,673 - modelscope - INFO - initiate model from location C:\Users\havei\.cache\modelscope\hub\models\iic\emotion2vec_base.
2025-11-20 12:37:31,678 - modelscope - INFO - initialize model from C:\Users\havei\.cache\modelscope\hub\models\iic\emotion2vec_base


New version available: 1.2.7. Your current version is 1.1.0.
Please use the command "pip install -U funasr" to upgrade.


2025-11-20 12:37:35,198 - modelscope - WARNING - No preprocessor field found in cfg.
2025-11-20 12:37:35,200 - modelscope - WARNING - No val key and type key found in preprocessor domain of configuration.json file.
2025-11-20 12:37:35,201 - modelscope - WARNING - Cannot find available config to build preprocessor at mode inference, current config: {'model_dir': 'C:\\Users\\havei\\.cache\\modelscope\\hub\\models\\iic\\emotion2vec_base'}. trying to build by task and model information.
2025-11-20 12:37:35,201 - modelscope - INFO - No preprocessor key ('funasr', 'emotion-recognition') found in PREPROCESSOR_MAP, skip building preprocessor. If the pipeline runs normally, please ignore this log.
2025-11-20 12:37:35,208 - modelscope - INFO - cuda is not available, using cpu instead.


Classifier loaded.


100%|██████████| 466/466 [1:41:06<00:00, 13.02s/it]


⭐ Test Accuracy = 91.32%
